### LÄSA IN DATA

In [51]:
print(X['age'].info())

<class 'pandas.core.series.Series'>
Index: 28328 entries, 0 to 29528
Series name: age
Non-Null Count  Dtype  
--------------  -----  
28307 non-null  float64
dtypes: float64(1)
memory usage: 442.6 KB
None


In [93]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# players_df = pd.read_csv('players.csv')
# appearances_df = pd.read_csv('appearances.csv')

# df = pd.merge(appearances_df, players_df, on='player_id', how='left')

# # print(df.head())
# # df.info()


# y = df['market_value_in_eur']
# X = df.drop(columns=['market_value_in_eur'])

# print(X.columns)

# features = ['goals', 'assists', 'minutes_played', 'position','']

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# 1. Läs in och summera
players = pd.read_csv('players.csv')
apps = pd.read_csv('appearances.csv')

stats = (
    apps.groupby('player_id')[['goals', 'assists', 'minutes_played']]
    .sum() 
    .reset_index()
)

# 2. Slå ihop och välj kolumner
df = pd.merge(players, stats, on='player_id').dropna(
    subset=['market_value_in_eur']
)
df['age'] = 2026 - pd.to_datetime(df['date_of_birth']).dt.year

features = [
    'position', 
    'age', 
    'goals', 
    'assists', 
    'minutes_played',
    'current_club_domestic_competition_id'
    ]

# 3. X och y (get_dummies gör om position direkt)
X = pd.get_dummies(df[features], drop_first=True) 

y = df['market_value_in_eur']



### EDA

### TRÄNA / TESTA

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

age_imputer = SimpleImputer(strategy='mean')       # eller 'median'
age_imputer.fit(X_train[['age']])                   # lär sig BARA från X_train

X_train['age'] = age_imputer.transform(X_train[['age']]) 
X_val['age']   = age_imputer.transform(X_val[['age']])
X_test['age']  = age_imputer.transform(X_test[['age']])

#Ny 
y_train_log = np.log1p(y_train)

params = {
    'max_depth': [None, 5, 10, 20, 30],
    'n_estimators': [50, 100, 150],
    'min_samples_split': [2, 5, 10]
}
clf = RandomForestRegressor(random_state=42)
gs = GridSearchCV(estimator=clf, param_grid=params, cv=2, n_jobs=-1, verbose=2)
gs.fit(X_train, y_train_log)
print("Bästa parametrar:", gs.best_params_) 
models = {
    "BaseLine" : DummyRegressor(strategy='mean'),
    'Linjär Regression' : LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    "Random Forest (Tuned)": gs.best_estimator_
}


for name, model in models.items():
    if name != "Random Forest (Tuned)":
        model.fit(X_train, y_train_log)

    # Prediktera och konvertera tillbaka till vanliga euro
    log_preds = model.predict(X_val)
    preds = np.expm1(log_preds)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    print(f'{name:20} | {rmse:12,.0f} € | {r2:6.3f}') ## <---------------------------------------------



# Träna på log(y) istället för y direkt
# y_train_log = np.log1p(y_train)
# model.fit(X_train, y_train_log)

# Transformera tillbaka gissningarna med expm1 innan du räknar RMSE

# preds = np.expm1(model.predict(X_test)).copu


Fitting 2 folds for each of 45 candidates, totalling 90 fits
[CV] END max_depth=None, min_samples_split=2, n_estimators=50; total time=   1.5s
[CV] END max_depth=None, min_samples_split=2, n_estimators=50; total time=   1.6s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.0s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.1s
[CV] END max_depth=None, min_samples_split=2, n_estimators=150; total time=   4.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=150; total time=   4.9s
[CV] END max_depth=None, min_samples_split=5, n_estimators=50; total time=   1.4s
[CV] END max_depth=None, min_samples_split=5, n_estimators=50; total time=   1.5s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=   2.7s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=   2.8s
[CV] END max_depth=None, min_samples_split=5, n_estimators=150; total time=   3.9s
[CV] END max_depth=None, min_s

### SPARA

In [109]:
test_pred_log = gs.best_estimator_.predict(X_test)
test_pred = np.expm1(test_pred_log)

final_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
final_r2 = r2_score(y_test, test_pred)

print('RMSE = ', final_rmse)
print('R2 = ', final_r2)


RMSE =  5321425.371663712
R2 =  0.5408159313864729


In [17]:
import joblib
modell = joblib.dump(models['Random Forest'], "rfr.pkl")

rfr = joblib.load("rfr.pkl")

In [16]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28328 entries, 0 to 29528
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   goals                28328 non-null  int64
 1   assists              28328 non-null  int64
 2   minutes_played       28328 non-null  int64
 3   position_Defender    28328 non-null  bool 
 4   position_Goalkeeper  28328 non-null  bool 
 5   position_Midfield    28328 non-null  bool 
 6   position_Missing     28328 non-null  bool 
dtypes: bool(4), int64(3)
memory usage: 995.9 KB


In [127]:
import numpy as np
import pandas as pd

# 1. Skapa en tom rad med exakt samma kolumnnamn som träningsdatan
calle = pd.DataFrame(0, index=[0], columns=X_train.columns)

# 2. Fyll i siffervärden
calle['age'] = 18
calle['goals'] = 1000
calle['assists'] = 9200
calle['minutes_played'] = 9200 * 90  # t.ex. en hel säsong som ordinarie

# 3. Sätt en 1:a på hans position (Anfallare / Attack)
# Beroende på hur texten ser ut i er data heter den oftast 'position_Attack'
for col in calle.columns:
    if 'position' in col and 'Attack' in col:
        calle[col] = 1

# 4. Sätt en 1:a på ligan (Allsvenskan = SE1)
for col in calle.columns:
    if 'SE1' in col:
        calle[col] = 1

# 5. Låt modellen gissa (kom ihåg att omvandla från log-skala med expm1!)
basta_modell = gs.best_estimator_
calle_log_pred = basta_modell.predict(calle)
calle_varde = np.expm1(calle_log_pred)[0]

print(f"Spelare: Calle Pålsson")
print(f"Stats:   22 mål, 22 assist, Allsvenskan")
print(f"Uppskattat marknadsvärde: {calle_varde:,.0f} €")

Spelare: Calle Pålsson
Stats:   22 mål, 22 assist, Allsvenskan
Uppskattat marknadsvärde: 89,044,181 €


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7b13cbce-e99b-4e65-bc2e-1168deb550cc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>